## Pointnet Model Testing

In [ ]:
import numpy as np
import pandas as pd
import onnxruntime as ort

from pointnet_train import TrainProfile

Importing packages...


2026-02-23 14:58:52.310838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Package import complete.


In [ ]:
CONFIG_FILE = "/mnt/d/models/kc46_lidar_lr001_94_2/final/kc46_lidar_lr001_94_2_config.json"
TEST_SET_PATH = '/mnt/d/kc46_sim_collect'
OUTPUT_PATH = '/mnt/d/kc46_sim_collect_with_truth'

pointnet = TrainProfile( CONFIG_FILE )

root - INFO - Training profile kc46_lidar_lr001_94_2_classification_pretrain already exists. Using existing profile...
root - INFO - The following datasets were found in data/kc46_lidar_lr001_94_2_classification_pretrain:
root - INFO - 	-> collect_2026.Jan.22_23.51.26.8719556.UTC	(not requested, but will included in training profile)
root - INFO - 	-> collect_2026.Jan.22_23.53.34.5658074.UTC	(not requested, but will included in training profile)
root - INFO - 	-> collect_2026.Jan.23_00.00.26.2538193.UTC	(not requested, but will included in training profile)
root - INFO - 	-> collect_2026.Jan.23_00.04.35.9506687.UTC	(not requested, but will included in training profile)
root - INFO - 	-> pc_set.joblib	
root - INFO - 
Datasets added successfully:

root - INFO - classification_pretrain
Random seed: 42
Class labels: dict_keys(['f-15_model', 'a-10', 'b-1b', 'b-2', 'c-5', 'c-12', 'c-17a', 'c-32', 'c-130j', 'e-3', 'f-15e', 'f-16', 'f-18e', 'f-22', 'g-iii', 'kc-46', 'kc-135', 'lj-25', 'mig-29'

In [ ]:
session = ort.InferenceSession( f"{pointnet._model_path}{pointnet._training_profiles['final']['path']}{pointnet._name}_final.onnx", providers = ['CUDAExecutionProvider'] )

I0000 00:00:1771876769.281625   18467 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6078 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:01:00.0, compute capability: 7.5
/mnt/e/repos/PointCloudProcessing/point_cloud_analysis/.venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 83 variables whereas the saved optimizer has 127 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
0it [00:00, ?it/s]2026-02-23 14:59:32.301550: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144
2026-02-23 14:59:35.186363: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d03d00d4f50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-23 14:59:35.186402: I external/local_xla/xla/service/service.cc:171]   StreamEx

In [4]:
if( result is not None ):    
    tp = {label: 0 for label in pointnet._part_labels}
    fp = {label: 0 for label in pointnet._part_labels}
    fn = {label: 0 for label in pointnet._part_labels}

    metrics = []
    for idx, label in enumerate( pointnet._part_labels ):
        is_true = (result['truth'] == idx)
        is_pred = (result['predictions'] == idx)

        tp[label] = np.sum( is_true & is_pred )
        fp[label] = np.sum( is_pred & ~is_true )
        fn[label] = np.sum( is_true & ~is_pred )

        metrics.append({
            'part': label,
            'precision': tp[label] / ( tp[label] + fp[label] ) if ( tp[label] + fp[label] ) > 0 else 0.0,
            'recall': tp[label] / ( tp[label] + fn[label] ) if ( tp[label] + fn[label] ) > 0 else 0.0
        })

    perf_df = pd.DataFrame( metrics )
    perf_df.set_index( 'part', inplace = True )

    display( perf_df )

,precision,recall
part,,
wing,0.158389,0.031405
fuselage,0.435773,0.859763
engine,0.072193,0.074647
hstab,0.000000,0.000000
vstab,0.000000,0.000000
landing_gear,0.000000,0.000000
armament,0.000000,0.000000
boom_wing,0.000000,0.000000
boom_hull,0.000000,0.000000
